# MAE 预训练模型重建可视化
每行代码均标注与训练代码的对应关系，确保推理逻辑与训练逻辑严格一致。

## Cell 1 — 参数配置
所有参数必须与训练时的 `--args` 完全一致，否则模型结构不匹配导致权重加载失败或重建结果失真。

In [ ]:
# ============================================================
# 【用户配置区】以下参数必须与训练命令行参数完全一致
# ============================================================

CHECKPOINT_PATH = './output_dir/checkpoint-99.pth'  # 预训练权重路径
IMAGE_PATH      = './test.jpg'                       # 任意测试图片
SAVE_PATH       = './vis_result.png'                 # 可视化保存路径

# --- 对应 main_pretrain.py: parser.add_argument('--img_size', default=224) ---
IMG_SIZE    = 224

# --- 对应 main_pretrain.py: parser.add_argument('--patch_size', default=16) ---
PATCH_SIZE  = 16

# --- 对应 main_pretrain.py: parser.add_argument('--encoder_dim', default=768) ---
ENCODER_DIM   = 768

# --- 对应 main_pretrain.py: parser.add_argument('--encoder_depth', default=16) ---
ENCODER_DEPTH = 16

# --- 对应 main_pretrain.py: parser.add_argument('--encoder_num_heads', default=16) ---
ENCODER_HEADS = 16

# --- 对应 main_pretrain.py: parser.add_argument('--decoder_dim', default=512) ---
DECODER_DIM   = 512

# --- 对应 main_pretrain.py: parser.add_argument('--decoder_depth', default=6) ---
DECODER_DEPTH = 6

# --- 对应 main_pretrain.py: parser.add_argument('--decoder_num_heads', default=16) ---
DECODER_HEADS = 16

# --- 对应 main_pretrain.py: parser.add_argument('--mask_ratio', default=0.75) ---
MASK_RATIO  = 0.75

# --- 对应 main_pretrain.py: parser.add_argument('--mask_type', default='random') ---
# 可选: 'random' / 'edge' / 'attn'
MASK_TYPE   = 'random'

# --- 对应 main_pretrain.py: parser.set_defaults(norm_pix_loss=False) ---
# ⚠️ 极其重要：必须与训练时 --norm_pix_loss 开关一致
# 若训练时加了 --norm_pix_loss 则设为 True，否则为 False
# 若与训练不符，mean/var 的计算路径不同，反归一化结果将完全错误
NORM_PIX_LOSS = False

## Cell 2 — 导入与设备设置

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms

# 导入自定义模型，与 main_pretrain.py 顶部 import 完全一致
from my_vit import MyVit
from my_mae import MyMaskedAutoencoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

## Cell 3 — 反归一化与 Unpatchify 工具函数

In [ ]:
def unpatchify(x, patch_size=16):
    """
    将模型输出的 patch 序列还原成完整图像张量。

    必须与 my_vit.py 的 patch_embed[0] (Rearrange) 严格互逆：
        patch_embed[0]: 'n c (x p1) (y p2) -> n (x y) (c p1 p2)'
    因此最后一维展开顺序是 (channel, row_pixel, col_pixel)，即 channel-first。
    还原步骤必须按此顺序 reshape，否则 RGB 通道会错位。

    输入 x : [B, L, patch_size**2 * C]   L = (H/P)*(W/P)
    输出   : [B, C, H, W]
    """
    B, L, _ = x.shape
    p = patch_size
    h = w = int(L ** 0.5)
    assert h * w == L, 'patch 数量必须是完全平方数（正方形图）'

    # 步骤1：还原 Rearrange 的最后一维 (c p1 p2) → [B, h, w, C, p, p]
    # 对应 patch_embed[0] 将 (c p1 p2) 展平的逆操作
    x = x.reshape(B, h, w, 3, p, p)

    # 步骤2：调整维度顺序，从 [B, h, w, C, p, p] → [B, C, h, p, w, p]
    # 对应 Rearrange 中 'n c (x p1) (y p2)' 的空间排列方式
    x = x.permute(0, 3, 1, 4, 2, 5).contiguous()

    # 步骤3：合并空间维度 [B, C, h, p, w, p] → [B, C, H, W]
    imgs = x.reshape(B, 3, h * p, w * p)
    return imgs


# ImageNet 归一化参数，与 main_pretrain.py transforms.Normalize 完全一致：
# transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def inverse_imagenet_normalize(t):
    """
    将 ImageNet 归一化后的 tensor 还原到 [0,1] 显示范围。

    训练时 main_pretrain.py 使用：
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    因此推理可视化时必须做对应的逆变换：
        pixel_display = pixel_normalized * std + mean

    输入 t: [B, C, H, W] 或 [C, H, W]，ImageNet 归一化后的值域（约 -2 ~ 2）
    输出  : 同形状，值域 [0, 1]（clip 处理数值误差）
    """
    mean = IMAGENET_MEAN.to(t.device)
    std  = IMAGENET_STD.to(t.device)
    return (t * std + mean).clamp(0, 1)

## Cell 4 — 加载模型

In [ ]:
# 模型实例化，参数与 main_pretrain.py 中完全一致
# 对应 main_pretrain.py:
#   encoder = MyVit(img_size=..., patch_size=..., embed_dim=..., depth=..., num_heads=...)
encoder = MyVit(
    img_size   = IMG_SIZE,
    patch_size = PATCH_SIZE,
    embed_dim  = ENCODER_DIM,
    depth      = ENCODER_DEPTH,
    num_heads  = ENCODER_HEADS
)

# 对应 main_pretrain.py:
#   model = MyMaskedAutoencoder(encoder=encoder, mask_type=..., mask_ratio=..., ...)
model = MyMaskedAutoencoder(
    encoder       = encoder,
    mask_type     = MASK_TYPE,
    mask_ratio    = MASK_RATIO,
    decoder_dim   = DECODER_DIM,
    decoder_depth = DECODER_DEPTH,
    decoder_num_heads = DECODER_HEADS,
    norm_pix_loss = NORM_PIX_LOSS    # ⚠️ 必须与训练时一致
)

# 加载权重
# misc.save_model 保存格式为 {'model': state_dict, 'optimizer': ..., 'epoch': ...}
# 对应 util/misc.py save_model 函数的保存格式
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu',weights_only=False)
state_dict = checkpoint['model']

# 处理 DDP 训练产生的 'module.' 前缀
# 若训练时使用了 DistributedDataParallel，权重 key 会带 'module.' 前缀
if any(k.startswith('module.') for k in state_dict.keys()):
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

msg = model.load_state_dict(state_dict, strict=True)  # strict=True 确保没有遗漏或多余的键
print(f'权重加载信息: {msg}')

model.to(device)
model.eval()  # 关闭 Dropout/BatchNorm 的训练模式
print('模型加载完成')

## Cell 5 — 图像预处理
必须完全复现 `main_pretrain.py` 的 `transform_train`，除了去掉随机裁剪和随机翻转这两个**数据增强**操作（推理时不需要随机性），**归一化参数必须保留**。

In [ ]:
# main_pretrain.py 的训练预处理：
#   transforms.RandomResizedCrop(img_size, ...)   ← 推理时用 Resize 替代，去掉随机性
#   transforms.RandomHorizontalFlip()             ← 推理时不需要
#   transforms.ToTensor()                         ← 保留，将 [0,255] uint8 转为 [0,1] float
#   transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  ← 必须保留！
#
# ⚠️ 归一化是模型训练数据分布的一部分，缺少此步骤会导致输入分布与训练不符，
#    进而使 forward_loss 中 mean/var 的计算值与训练时不一致，重建颜色完全偏移。

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),   # 固定尺寸，对应训练的 img_size=224
    transforms.ToTensor(),                     # [H,W,C] uint8 → [C,H,W] float32, 值域 [0,1]
    transforms.Normalize(                      # ImageNet 归一化，与训练完全一致
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

img_pil = Image.open(IMAGE_PATH).convert('RGB')  # 确保 3 通道，对应 in_chans=3

# [C, H, W] → [1, C, H, W]，增加 batch 维度
x = transform(img_pil).unsqueeze(0).to(device)

print(f'输入 tensor shape: {x.shape}')   # 应为 [1, 3, 224, 224]
print(f'输入值域: [{x.min():.3f}, {x.max():.3f}]')  # 归一化后约为 [-2.1, 2.6]

## Cell 6 — 前向推理与反归一化
这是最容易出错的环节，逐步对应 `my_mae.py` 的 `forward` 和 `forward_loss`。

In [ ]:
with torch.no_grad():
    # my_mae.py forward() 返回: loss, mean, var, pred, mask
    #
    # loss : 标量，masked patch 的平均重建 loss
    # mean : [B, L, 1]  每个 patch 的像素均值（ImageNet归一化空间）
    #         对应 forward_loss: mean = target.mean(dim=-1, keepdim=True)
    #         若 norm_pix_loss=False，则 mean 全为 0
    # var  : [B, L, 1]  每个 patch 的像素方差
    #         对应 forward_loss: var  = target.var(dim=-1, keepdim=True)
    #         若 norm_pix_loss=False，则 var  全为 1
    # pred : [B, L, patch_size**2 * 3]  模型预测的 patch 像素
    #         若 norm_pix_loss=True，pred 在「patch 标准化空间」中
    #         若 norm_pix_loss=False，pred 直接在「ImageNet 归一化空间」中
    # mask : [B, L]  0=保留(visible), 1=被遮盖(masked)
    #         对应 random_masking/edge_guided_masking 中 mask 的构造逻辑
    loss, mean, var, pred, mask = model(x)

print(f'重建 loss : {loss.item():.6f}')
print(f'pred shape: {pred.shape}')   # [1, 196, 768]  (196=14*14, 768=16*16*3)
print(f'mask shape: {mask.shape}')   # [1, 196]
print(f'masked patch 比例: {mask.float().mean().item():.2%}')  # 应接近 MASK_RATIO

# ------------------------------------------------------------
# 步骤 A：反「patch 标准化」（对应 forward_loss 中的 norm_pix_loss 分支）
#
# 训练时若 norm_pix_loss=True：
#   target = (target - mean) / (var + 1e-6)^0.5
#   模型学习预测标准化后的 target，因此 pred 也在标准化空间
# 逆变换：pred_unnorm = pred * (var + 1e-6)^0.5 + mean
#
# 若 norm_pix_loss=False：mean=0, var=1，此步为恒等变换，不影响结果
# 两种情况公式相同，可统一处理。
# ------------------------------------------------------------
pred_unnorm = pred * (var + 1.e-6) ** 0.5 + mean
# pred_unnorm: [B, L, patch_size**2 * 3]，现在在「ImageNet 归一化空间」

# ------------------------------------------------------------
# 步骤 B：将 patch 序列还原为图像 tensor
# unpatchify 必须与 my_vit.py patch_embed[0] 的 Rearrange 完全互逆
# ------------------------------------------------------------
pred_img = unpatchify(pred_unnorm, PATCH_SIZE)   # [1, 3, 224, 224]，ImageNet 归一化空间

# ------------------------------------------------------------
# 步骤 C：反 ImageNet 归一化，还原到 [0,1] 显示空间
# 对应 main_pretrain.py transforms.Normalize 的逆操作
# 原图 x 同样需要反归一化才能正确显示
# ------------------------------------------------------------
original_display = inverse_imagenet_normalize(x[0])     # [3, 224, 224], [0,1]
pred_display     = inverse_imagenet_normalize(pred_img[0])  # [3, 224, 224], [0,1]

# numpy 格式，用于 matplotlib 显示
original_np = original_display.cpu().permute(1, 2, 0).numpy()  # [H, W, 3]
pred_np     = pred_display.cpu().permute(1, 2, 0).numpy()      # [H, W, 3]

## Cell 7 — 生成 Masked 图（保留区域+灰色遮盖）

In [ ]:
# mask: [1, L]，0=保留，1=被遮盖
# 将 mask 扩展到像素级别

# mask [1, L] → [1, L, 1] → [1, L, patch_size^2 * 3] (每个 patch 内所有像素同一个 mask 值)
mask_patches = mask.unsqueeze(-1).repeat(1, 1, PATCH_SIZE ** 2 * 3)  # [1, L, P^2*3]

# unpatchify → [1, 3, 224, 224]，像素级 mask（0=保留，1=遮盖）
mask_img = unpatchify(mask_patches.float(), PATCH_SIZE)  # [1, 3, 224, 224]
mask_np  = mask_img[0].cpu().permute(1, 2, 0).numpy()   # [H, W, 3]

# 合成 masked 图：保留区域显示原图，遮盖区域显示灰色（0.5）
# 0.5 对应的是显示空间的中性灰色，与图示效果一致
im_masked = original_np * (1 - mask_np) + 0.5 * mask_np

# 合成重建图：保留区域用原图像素，遮盖区域用模型预测
# 这与训练 loss 的计算范围一致：loss 只对 mask==1 的区域计算
# 对应 forward_loss: loss = (loss * mask).sum() / mask.sum()
im_paste = original_np * (1 - mask_np) + pred_np * mask_np

# clip 防止浮点误差超出显示范围
original_np = np.clip(original_np, 0, 1)
im_masked   = np.clip(im_masked,   0, 1)
im_paste    = np.clip(im_paste,    0, 1)

## Cell 8 — 可视化输出

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(original_np)
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

axes[1].imshow(im_masked)
axes[1].set_title(f'Masked ({MASK_RATIO*100:.0f}%) - {MASK_TYPE.capitalize()}', fontsize=12)
axes[1].axis('off')

axes[2].imshow(im_paste)
axes[2].set_title('Reconstructed', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'已保存至 {SAVE_PATH}')

## Cell 9 — （可选）批量测试多张图片

In [ ]:
import os

def reconstruct_and_show(image_path, model, transform, device,
                          patch_size=16, mask_ratio=0.75, mask_type='random',
                          save_path=None):
    """
    单张图片重建流程的封装函数。
    所有步骤与 Cell 5-8 完全一致，方便批量调用。
    """
    # 预处理（同 Cell 5）
    img_pil = Image.open(image_path).convert('RGB')
    x = transform(img_pil).unsqueeze(0).to(device)

    # 推理（同 Cell 6）
    with torch.no_grad():
        loss, mean, var, pred, mask = model(x)

    # 反归一化（同 Cell 6）
    pred_unnorm  = pred * (var + 1.e-6) ** 0.5 + mean
    pred_img     = unpatchify(pred_unnorm, patch_size)
    original_display = inverse_imagenet_normalize(x[0])
    pred_display     = inverse_imagenet_normalize(pred_img[0])
    original_np  = original_display.cpu().permute(1, 2, 0).numpy()
    pred_np      = pred_display.cpu().permute(1, 2, 0).numpy()

    # Masked 图（同 Cell 7）
    mask_patches = mask.unsqueeze(-1).repeat(1, 1, patch_size ** 2 * 3)
    mask_img     = unpatchify(mask_patches.float(), patch_size)
    mask_np      = mask_img[0].cpu().permute(1, 2, 0).numpy()
    im_masked    = original_np * (1 - mask_np) + 0.5 * mask_np
    im_paste     = original_np * (1 - mask_np) + pred_np * mask_np

    original_np = np.clip(original_np, 0, 1)
    im_masked   = np.clip(im_masked,   0, 1)
    im_paste    = np.clip(im_paste,    0, 1)

    # 可视化
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    fig.suptitle(os.path.basename(image_path), fontsize=10, color='gray')
    axes[0].imshow(original_np);  axes[0].set_title('Original');       axes[0].axis('off')
    axes[1].imshow(im_masked);    axes[1].set_title(f'Masked ({mask_ratio*100:.0f}%) - {mask_type}'); axes[1].axis('off')
    axes[2].imshow(im_paste);     axes[2].set_title('Reconstructed');  axes[2].axis('off')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'loss={loss.item():.6f}  masked={mask.float().mean().item():.2%}')


# 批量示例
test_images = [
    './test1.jpg',
    './test2.jpg',
]

for img_path in test_images:
    if os.path.exists(img_path):
        reconstruct_and_show(
            img_path, model, transform, device,
            patch_size=PATCH_SIZE, mask_ratio=MASK_RATIO, mask_type=MASK_TYPE
        )